In [17]:
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [2]:
load_dotenv()

embedding_model = GoogleGenerativeAIEmbeddings(
    model = 'gemini-embedding-2-preview'
)

model = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite-preview'
)

In [3]:
loader = WikipediaLoader(
    query = '10 avatars of lord vishnu',
    lang = 'en' , 
    load_max_docs=4
)

In [4]:
doc = loader.load()

In [5]:
doc

[Document(metadata={'title': 'Dashavatara', 'summary': 'The Dashavatara (Sanskrit: दशावतार, IAST: daśāvatāra) are the ten primary avatars of Vishnu, a principal Hindu god. Vishnu is said to descend in the form of an avatar to restore cosmic order. The word Dashavatara derives from daśa, meaning "ten", and avatāra, roughly equivalent to "incarnation".\nThe list of included avatars varies across sects and regions, particularly with respect to the inclusion of Balarama (brother of Krishna) or the Buddha. Though no list can be uncontroversially presented as standard, the "most accepted list found in Puranas and other texts is [...] Krishna, Buddha." Most draw from the following set of figures, in this order: Matsya, Kurma, Varaha, Narasimha, Vamana, Parashurama, Rama, Krishna, or Balarama, Buddha or Krishna, and Kalki. In traditions that omit Krishna, he often replaces Vishnu as the source of all avatars. Some traditions include a regional deity such as Vithoba or Jagannath in penultimate 

In [6]:
for i in range(len(doc)):
    print('The character in the doc is :' , len(doc[i].page_content))

The character in the doc is : 4000
The character in the doc is : 4000
The character in the doc is : 4000
The character in the doc is : 4000


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000 , 
    chunk_overlap = 200
)

In [8]:
document = splitter.split_documents(doc)
print(len(document))

24


In [9]:
chroma_store = Chroma.from_documents(
    embedding=embedding_model , 
    documents = document
)

In [10]:
chroma_store.get(include=['embeddings'])['embeddings']

array([[-0.0017134 ,  0.01482703,  0.01979922, ...,  0.00080581,
        -0.00272957, -0.00619585],
       [-0.00333399,  0.0149163 ,  0.01719606, ..., -0.00656068,
         0.00828494,  0.0024656 ],
       [-0.00626822,  0.01691827,  0.0102591 , ..., -0.00940543,
         0.00330306,  0.00172835],
       ...,
       [ 0.00200766,  0.03140792,  0.01612656, ..., -0.01990714,
         0.00196664, -0.00769115],
       [ 0.00360506,  0.02499723,  0.01967558, ..., -0.01806792,
        -0.00737189, -0.01235972],
       [ 0.00369317,  0.03817554,  0.0053473 , ..., -0.01956276,
        -0.00153585, -0.00484728]], shape=(24, 3072))

In [11]:
semantic = chroma_store.similarity_search_with_score(
    query = 'Maryada Purshottam', 
    k = 4
)

In [12]:
semantic

[(Document(id='a1386a70-1417-4ebf-9ce9-1d55605d14b4', metadata={'title': 'Dashavatara', 'summary': 'The Dashavatara (Sanskrit: दशावतार, IAST: daśāvatāra) are the ten primary avatars of Vishnu, a principal Hindu god. Vishnu is said to descend in the form of an avatar to restore cosmic order. The word Dashavatara derives from daśa, meaning "ten", and avatāra, roughly equivalent to "incarnation".\nThe list of included avatars varies across sects and regions, particularly with respect to the inclusion of Balarama (brother of Krishna) or the Buddha. Though no list can be uncontroversially presented as standard, the "most accepted list found in Puranas and other texts is [...] Krishna, Buddha." Most draw from the following set of figures, in this order: Matsya, Kurma, Varaha, Narasimha, Vamana, Parashurama, Rama, Krishna, or Balarama, Buddha or Krishna, and Kalki. In traditions that omit Krishna, he often replaces Vishnu as the source of all avatars. Some traditions include a regional deity 

In [13]:
retriver = chroma_store.as_retriever(
    search_type = 'similarity' , 
    search_kwargs = {'k' : 4}
)

In [14]:
semantic

[(Document(id='a1386a70-1417-4ebf-9ce9-1d55605d14b4', metadata={'title': 'Dashavatara', 'summary': 'The Dashavatara (Sanskrit: दशावतार, IAST: daśāvatāra) are the ten primary avatars of Vishnu, a principal Hindu god. Vishnu is said to descend in the form of an avatar to restore cosmic order. The word Dashavatara derives from daśa, meaning "ten", and avatāra, roughly equivalent to "incarnation".\nThe list of included avatars varies across sects and regions, particularly with respect to the inclusion of Balarama (brother of Krishna) or the Buddha. Though no list can be uncontroversially presented as standard, the "most accepted list found in Puranas and other texts is [...] Krishna, Buddha." Most draw from the following set of figures, in this order: Matsya, Kurma, Varaha, Narasimha, Vamana, Parashurama, Rama, Krishna, or Balarama, Buddha or Krishna, and Kalki. In traditions that omit Krishna, he often replaces Vishnu as the source of all avatars. Some traditions include a regional deity 

In [15]:
retriver.invoke('maryada Purshottam' , 
)[0].metadata['summary']

'The Dashavatara (Sanskrit: दशावतार, IAST: daśāvatāra) are the ten primary avatars of Vishnu, a principal Hindu god. Vishnu is said to descend in the form of an avatar to restore cosmic order. The word Dashavatara derives from daśa, meaning "ten", and avatāra, roughly equivalent to "incarnation".\nThe list of included avatars varies across sects and regions, particularly with respect to the inclusion of Balarama (brother of Krishna) or the Buddha. Though no list can be uncontroversially presented as standard, the "most accepted list found in Puranas and other texts is [...] Krishna, Buddha." Most draw from the following set of figures, in this order: Matsya, Kurma, Varaha, Narasimha, Vamana, Parashurama, Rama, Krishna, or Balarama, Buddha or Krishna, and Kalki. In traditions that omit Krishna, he often replaces Vishnu as the source of all avatars. Some traditions include a regional deity such as Vithoba or Jagannath in penultimate position, replacing Krishna or Buddha. All avatars have

In [16]:
user_query = 'What are 10 avatars of lord vishnu'

In [ ]:
prompt = PromptTemplate(
    template='You are my assistant and you have to answer from the given context \n if the context is insufficient then just give i don\'t know \n context -- > {context_text} \n  user_query -- > {query}', 
    input_variables=['context_text' , 'query']
)

In [20]:
retrive_doc = retriver.invoke(user_query)

In [22]:
context_text = '\n\n'.join(doc.page_content for doc in retrive_doc)

In [26]:
chain = prompt |  model 

result = chain.invoke({'context_text':context_text , 'query' : user_query })

In [27]:
result.text

'Based on the provided context, the list of avatars varies across sects and regions, and no single list is universally standard. However, the text mentions that the figures are generally drawn from the following set: \n\nMatsya, Kurma, Varaha, Narasimha, Vamana, Parashurama, Rama, Krishna (or Balarama), Buddha (or Krishna), and Kalki.'